# Get Nadex Contract Locations

Downloads daily settlement PDFs from Nadex's public S3 bucket, extracts US 500 binary
contract strikes, and identifies the strikes that bracket each day's 09:30 market open.

**Output:** `contract_locations.csv` — columns: `date`, `above`, `below`

**Run before `barsToCleaning.ipynb`** whenever new trading dates have been added to `GoodOldGoodOld.csv`.

> PDFs are settlement results, available after market close.  
> Same-day (live) contract lookup is not supported by this script.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Filters to isolate US 500 daily binary options expiring at 4:15 PM ET
PRODUCT_FILTER = "US 500"
PERIOD_FILTER  = "Daily"
EXPIRY_FILTER  = "4:15PM"

PDF_URL     = "https://s3.amazonaws.com/market-data-prod.nadex.com/{date}_tradingResults.pdf"
OUTPUT_FILE = "contract_locations.csv"
BARS_FILE   = "GoodOldGoodOld.csv"
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import re
import io
import logging
import requests
import pandas as pd
import pdfplumber

# Suppress harmless pdfminer decompression warnings (noise from PDF internal structure)
logging.getLogger('pdfminer').setLevel(logging.ERROR)

In [ ]:
# Load 1-minute bars and extract the 09:30:00 open price for each trading day
bars = pd.read_csv(BARS_FILE)
bars['date_only'] = pd.to_datetime(bars['date']).dt.normalize()

opens = (
    bars[bars['time'] == '09:30:00']
    .groupby('date_only')['open']
    .first()
    .reset_index()
    .rename(columns={'date_only': 'date', 'open': 'market_open'})
)

print(f"Trading days in {BARS_FILE}: {len(opens)}")
print(opens.tail())

In [ ]:
# Load existing contract_locations.csv for incremental updates.
# Dates already covered are skipped — no redundant PDF downloads.
try:
    existing = pd.read_csv(OUTPUT_FILE, parse_dates=['date'])
    existing['date'] = existing['date'].dt.normalize()
    covered_dates = set(existing['date'])
    print(f"Existing {OUTPUT_FILE} covers {len(covered_dates)} date(s)")
except FileNotFoundError:
    existing = pd.DataFrame(columns=['date', 'above', 'below'])
    covered_dates = set()
    print(f"No existing {OUTPUT_FILE} — building from scratch")

dates_to_fetch = [d for d in opens['date'] if d not in covered_dates]
print(f"Dates to fetch: {len(dates_to_fetch)}")

In [ ]:
# Regex to extract strike from Display Name.
# Handles both '>' and '+' prefix formats, e.g.:
#   "US 500 (Jun) >5004.0 (4:15PM)"
#   "US 500 (Jun) +4823.6 (4:15PM)"
_STRIKE_RE = re.compile(r'US 500[^>+\d]*[>+]([\d]{4,5}(?:\.\d+)?)')


def _parse_strikes_from_pdf(pdf_bytes):
    """Extract US 500 daily binary strike prices from a Nadex results PDF bytes.
    Uses full-text regex scan — fast and sufficient since the regex is precise.
    """
    strikes = []
    with pdfplumber.open(pdf_bytes) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ''
            for line in text.splitlines():
                if PRODUCT_FILTER not in line or EXPIRY_FILTER not in line:
                    continue
                m = _STRIKE_RE.search(line)
                if m:
                    strikes.append(float(m.group(1)))
    return strikes


def get_contracts_for_date(date, market_open):
    """Download and parse the Nadex results PDF for `date`.

    Returns (above_strike, below_strike) — the US 500 daily binary contract
    strikes immediately above and below `market_open`.
    Returns (None, None) if the PDF is unavailable (weekend / holiday /
    future date) or a network error occurs.
    """
    url = PDF_URL.format(date=date.strftime('%Y%m%d'))
    try:
        resp = requests.get(url, timeout=30)
    except requests.RequestException as e:
        print(f'  Network error: {e}')
        return None, None

    if resp.status_code != 200:
        return None, None  # Weekend, holiday, or PDF not yet published

    strikes = _parse_strikes_from_pdf(io.BytesIO(resp.content))
    if not strikes:
        print(f'  WARNING: PDF downloaded but no US 500 daily strikes found')
        return None, None

    strikes = sorted(set(strikes))
    belows  = [s for s in strikes if s <= market_open]
    aboves  = [s for s in strikes if s >  market_open]
    below   = max(belows) if belows else None
    above   = min(aboves) if aboves else None
    return above, below

In [ ]:
# Fetch contract data for all missing dates and save to contract_locations.csv.
# Progress is printed per date. Dates with no PDF are holidays/weekends.
rows = []
for date in dates_to_fetch:
    market_open_vals = opens.loc[opens['date'] == date, 'market_open'].values
    if len(market_open_vals) == 0:
        continue
    market_open = market_open_vals[0]

    above, below = get_contracts_for_date(date, market_open)

    if above is not None or below is not None:
        rows.append({'date': date, 'above': above, 'below': below})
        print(f"{date.date()}  open={market_open:.2f}  below={below}  above={above}")
    else:
        print(f"{date.date()}  — no data (holiday / weekend / future date)")

new_data = pd.DataFrame(rows)
combined = (
    pd.concat([existing, new_data], ignore_index=True)
    .drop_duplicates('date')
    .sort_values('date')
    .reset_index(drop=True)
)
combined.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved {len(combined)} date(s) to {OUTPUT_FILE}")